# 🔍 Module 12: Production Observability, Debugging & Common Pitfalls

In this capstone module, we review the top production bugs that bring down async systems, and how to debug them like a senior engineer.

### What We Will Master:
1. **Enabling Debug Mode**: `asyncio.run(main(), debug=True)` & `loop.slow_callback_duration`.
2. **Event Loop Lag Monitoring**: Detecting when blocking code freezes your server.
3. **Detecting Unawaited Coroutines & Unhandled Task Exceptions**.
4. **The Top 7 Fatal Async Pitfalls in Python**.
5. **The Senior Engineer's Production Async Checklist**.

In [1]:
import asyncio
import time

async def monitor_event_loop_lag(threshold_ms: float = 50.0):
    print("💓 Heartbeat monitor starting...")
    for _ in range(2):
        t0 = time.perf_counter()
        await asyncio.sleep(0.1)
        lag = ((time.perf_counter() - t0) - 0.1) * 1000
        print(f"Loop lag: {lag:.1f}ms (Healthy)")
    
    # Simulate accidental blocking code
    t0 = time.perf_counter()
    time.sleep(0.3) # 🛑 ACCIDENTAL BLOCKING CALL!
    await asyncio.sleep(0.1)
    lag = ((time.perf_counter() - t0) - 0.1) * 1000
    print(f"⚠️ ALERT! Event Loop lag spiked to {lag:.1f}ms! (Blocking CPU code detected!)")
    
    t0 = time.perf_counter()
    await asyncio.sleep(0.1)
    lag = ((time.perf_counter() - t0) - 0.1) * 1000
    print(f"Loop lag: {lag:.1f}ms (Recovered)")

await monitor_event_loop_lag()

💓 Heartbeat monitor starting...
Loop lag: 0.8ms (Healthy)
Loop lag: 1.1ms (Healthy)
⚠️ ALERT! Event Loop lag spiked to 312.4ms! (Blocking CPU code detected!)
Loop lag: 0.9ms (Recovered)


---
## 2. The Top 7 Fatal Async Python Pitfalls in Production

| # | The Pitfall | Why It Kills Production | The Fix |
|---|---|---|---|
| 1 | **Blocking the Loop** | `time.sleep()`, `requests.get()`, heavy loops freeze all users | Use `asyncio.to_thread` or `ProcessPoolExecutor` |
| 2 | **Forgetting `await`** | Coroutine object returned; code never executes | Run with `PYTHONASYNCIODEBUG=1` to see warnings |
| 3 | **Unreferenced `create_task`** | Python GC destroys active background tasks mid-execution | Keep tasks in a strong reference `set()` |
| 4 | **Unhandled Task Exceptions** | Tasks crash silently without logging | Use `asyncio.TaskGroup` or add exception callbacks |
| 5 | **Race Conditions** | Shared state modified across `await` points | Protect mutable shared state with `asyncio.Lock` |
| 6 | **FastAPI `async def` on CPU Work** | Blocks worker threadpool | Use regular `def` in FastAPI for blocking work |
| 7 | **Nested `asyncio.run()`** | Raises `RuntimeError: This event loop is already running` | In async code, use `await` or `create_task`, never `run()` |

---
## 🏆 The Senior Engineer's Production Async Checklist

- [x] **Zero Loop Blockage**: All CPU math & sync SDKs run in thread/process pools.
- [x] **Structured Concurrency**: Using `asyncio.TaskGroup` over raw `gather` to avoid orphaned tasks.
- [x] **Reused Connection Pools**: Database & HTTP clients share pools with concurrency limits.
- [x] **API Rate Limiting**: Outgoing calls to external vendors protected by `asyncio.Semaphore`.
- [x] **Timeouts on Every Network Call**: Using `asyncio.timeout` with fallback cached responses.
- [x] **Graceful Shutdown**: All queues drained and `CancelledError` caught for zero data loss.
- [x] **Loop Lag Telemetry**: Real-time heartbeat alerts if loop lag exceeds 50ms.

🎉 **Congratulations! You have completed the entire Async Python Curriculum!**